# SQL Analytics

## Objective


Use the dimensional MySQL model developed in the previous stages to answer the project's business questions through analytical SQL queries.

This stage focuses on descriptive and exploratory analysis of well production, hydraulic fracturing, geological, operational, temporal, and spatial data.

The analysis follows the structure:

**Business Question → SQL Query → Result → Interpretation → Business Conclusion**

## Analytical Scope

The SQL analysis will address the business questions defined in the Business Understanding stage.

The analysis is organized into six analytical areas:

1. Production Performance
2. Temporal Analysis
3. Hydraulic Fracturing
4. Geological Analysis
5. Operational Analysis
6. Spatial Analysis

The analysis will use the dimensional model already implemented in MySQL.

No additional data loading, ETL, or modification of the fact tables will be performed in this stage.

## Dimensional Model

The SQL analysis will be performed using the following dimensional model:

### Fact Tables

- `fact_production`
- `fact_fracturing`

### Dimension Tables

- `dim_time`
- `dim_well`
- `dim_geology`
- `dim_field_area`
- `dim_location`
- `dim_company`

The fact tables contain the analytical measurements, while the dimension tables provide the temporal, well, geological, spatial, and company attributes required to analyze production and fracturing performance.

## Analytical Principles

The SQL analysis will follow these principles:

- Use only variables available in the dimensional model.
- Preserve the definitions established during the Business Understanding stage.
- Prefer transparent and reproducible SQL queries.
- Distinguish descriptive patterns from causal relationships.
- Report limitations when data coverage or completeness affects interpretation.
- Avoid unsupported assumptions or artificial imputation.
- Validate analytical results before drawing business conclusions.

The purpose of this stage is to identify patterns and associations that can later be investigated through Python-based statistical analysis.

## MySQL Connection

In [2]:
import pandas as pd
import mysql.connector

In [3]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)
cursor.execute("USE oil_gas_dw")

print("Database selected.")

Database selected.


## BQ-01 — How does well production evolve over time?

### Business Question

How does well production evolve over time?

### Analytical Approach

Production will be aggregated monthly to evaluate the temporal evolution of oil and gas production.

The production fact table will be joined with the time dimension through `time_key`.

In [5]:
query_bq01 = """
SELECT
    t.year,
    t.month,
    SUM(fp.prod_pet) AS total_oil_production,
    SUM(fp.prod_gas) AS total_gas_production
FROM fact_production AS fp
JOIN dim_time AS t
    ON fp.time_key = t.time_key
GROUP BY
    t.year,
    t.month
ORDER BY
    t.year,
    t.month;
"""

In [6]:
df_bq01 = pd.read_sql(query_bq01, conn)

df_bq01.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_2404\2014480578.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq01 = pd.read_sql(query_bq01, conn)


,year,month,total_oil_production,total_gas_production
0,2006,1,1070.01,32508.00
1,2006,2,994.52,30792.89
2,2006,3,1136.85,33042.57
3,2006,4,1113.70,31362.11
4,2006,5,1505.22,39218.74


Check 1: Month of maximum and minimum production

In [9]:
query_control_01 = """
SELECT
    min(t.full_date),
    max(t.full_date)
FROM dim_time AS t
JOIN fact_production AS fp
    ON fp.time_key = t.time_key
"""

df_control_01 = pd.read_sql(query_control_01, conn)
df_control_01

C:\Users\Admin\AppData\Local\Temp\ipykernel_2404\2134265284.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_control_01 = pd.read_sql(query_control_01, conn)


,min(t.full_date),max(t.full_date)
0,2006-01-31,2026-06-30


Check 2: months with no production records

In [12]:
query_control_02 = """
SELECT
    cal.year,
    cal.month
FROM (
    SELECT DISTINCT
        year,
        month
    FROM dim_time
    WHERE full_date <= (
        SELECT MAX(t2.full_date)
        FROM dim_time AS t2
        JOIN fact_production AS fp2
            ON t2.time_key = fp2.time_key
    )
) AS cal
LEFT JOIN (
    SELECT DISTINCT
        t.year,
        t.month
    FROM fact_production AS fp
    JOIN dim_time AS t
        ON fp.time_key = t.time_key
) AS prod
    ON cal.year = prod.year
    AND cal.month = prod.month
WHERE prod.year IS NULL
ORDER BY
    cal.year,
    cal.month;
"""

df_control_02 = pd.read_sql(query_control_02, conn)
df_control_02

C:\Users\Admin\AppData\Local\Temp\ipykernel_2404\2095310745.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_control_02 = pd.read_sql(query_control_02, conn)


,year,month
